# Lean 17c — Le lake `knot_lean` par ses déclarations (compagnon formel)

Compagnon du **Lean-17** (« Conway, les Nœuds et la Preuve de Piccirillo »). Là où Lean-17 raconte l'histoire mathématique (Fox 3-colorabilité, mutants, Piccirillo 2020, Lidman 2026) et interroge `Conway.lean` et `Lidman.lean`, ce notebook interroge les modules que Lean-17 ne cite pas — **`Basic.lean`** (les structures fondamentales), **`Invariant.lean`** (la chaîne 3-colorabilité, 71 déclarations), **`Reidemeister.lean`** (les moves R1/R2/R3 et leurs murs) et **`MathlibPrerequisites.lean`** (la feuille de route des 11 prérequis Mathlib) — et mesure l'état de la formalisation : déclarations par module, sorries réels vs prose, axiomes admis.

Kernel Python par choix : le kernel natif `lean4-wsl` est gelé en attendant la mise à jour du binaire `repl` (#11874 — construit pour v4.30.0, lakes en v4.32.1). La lecture des sources reste réelle : chaque cellule ouvre le fichier `.lean` du lake et en extrait les déclarations, jamais une copie.

Voir `knot_lean/README.md` pour l'état détaillé du corridor Reidemeister.

## 1. Le lac : sept fichiers, deux langues

Le lake suit la convention i18n #4980 : chaque module français a son sibling `_en` byte-identique sur le code, docstrings traduites. On inventorie les six modules FR et leurs miroirs.

In [1]:
import re
from pathlib import Path

LAKE = Path('knot_lean')
modules = sorted(p for p in (LAKE / 'Knots').glob('*.lean') if not p.stem.endswith('_en'))
print(f"Modules FR du lake : {len(modules)}")
for p in modules:
    en = p.with_name(p.stem + '_en.lean')
    sib = 'oui' if en.exists() else 'ABSENT'
    print(f"  {p.name:28} {len(p.read_text(encoding='utf-8').splitlines()):>5} lignes   sibling _en : {sib}")

Modules FR du lake : 6
  Basic.lean                     653 lignes   sibling _en : oui
  Conway.lean                    249 lignes   sibling _en : oui
  Invariant.lean                3540 lignes   sibling _en : oui
  Lidman.lean                    144 lignes   sibling _en : oui
  MathlibPrerequisites.lean      138 lignes   sibling _en : oui
  Reidemeister.lean             1069 lignes   sibling _en : oui


### Lecture du résultat
Six modules FR, chacun avec son miroir `_en` — la convention i18n est respectée à 100 % sur ce lake. `MathlibPrerequisites` est le plus gros en lignes (cadre des prérequis), `Invariant.lean` porte le plus de déclarations.

## 2. `Basic.lean` — les structures fondamentales

Tout le lake repose sur quatre structures : `Crossing`, `PDCrossing`, `KnotDiagram`, `Knot` (+ `Link`). Les nœuds canoniques (`unknot`, `trefoil`, `figureEight`) y sont définis, ainsi que le miroir et le nombre de croisements.

In [2]:
# Declarations publiques de Basic.lean, extraites du fichier reel
DECL_RE = re.compile(r'^(structure|theorem|lemma|def|instance|abbrev)\s+([A-Za-z_][A-Za-z0-9_.\'\?]*)')

def decls_of(path):
    out = []
    for i, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        m = DECL_RE.match(line)
        if m:
            out.append((i, m.group(1), m.group(2)))
    return out

basic = decls_of(LAKE / 'Knots' / 'Basic.lean')
print(f"Basic.lean : {len(basic)} declarations")
for ln, kind, name in basic:
    print(f"  L{ln:>4}  {kind:9} {name}")

Basic.lean : 34 declarations
  L  53  structure Crossing
  L  81  structure PDCrossing
  L  90  structure KnotDiagram
  L 109  structure Knot
  L 119  structure Link
  L 127  def       Knot.toLink
  L 137  def       unknotDiagram
  L 141  def       unknot
  L 149  def       trefoilDiagram
  L 157  def       trefoil
  L 164  def       figureEightDiagram
  L 173  def       figureEight
  L 181  def       mirrorCrossing
  L 187  def       Knot.mirror
  L 199  def       Knot.crossingNumberOfDiagram
  L 225  def       Knot.crossingNumber
  L 234  def       KnotDiagram.edges
  L 238  def       KnotDiagram.numCrossings
  L 275  def       KnotDiagram.wf
  L 284  theorem   unknot_wf
  L 288  theorem   trefoil_wf
  L 292  theorem   figureEight_wf
  L 316  theorem   mirror_unknot_wf
  L 323  theorem   mirror_trefoil_wf
  L 329  theorem   mirror_figureEight_wf
  L 365  theorem   mirrorCrossing_perm
  L 386  theorem   mirrorCrossing_preserves_count
  L 397  theorem   count_lift_append
  L 422  theor

### Lecture du résultat
Les trois nœuds canoniques du lake sont des `def` explicites (`unknotDiagram`, `trefoilDiagram`, `figureEightDiagram` puis leurs objets `Knot`). `wf` (well-formed) est un `Bool` : la correction du PD-code est décidable et testée — `unknot_wf` en est la preuve minimale.

## 3. `Invariant.lean` — la chaîne 3-colorabilité (71 déclarations)

C'est le cœur pédagogique du lake : la définition `IsTricolorable` et le théorème-cible `tricolorable_invariant` (« la 3-colorabilité de Fox est invariante par moves de Reidemeister »), avec sa décomposition en bras ascendants R1/R2 et ses murs nommés.

In [3]:
# Les declarations cibles de la chaine tricolorabilite
inv = decls_of(LAKE / 'Knots' / 'Invariant.lean')
print(f"Invariant.lean : {len(inv)} declarations\n")
CIBLES = ['IsTricolorable', 'triColorFoxCondition_iff_sum_mod_three',
          'tricolorable_invariant', 'tricolorable_forward_r1',
          'tricolorable_forward_r2_up', 'r2_append_only_wall', 'r3_determined_wall']
for ln, kind, name in inv:
    if any(name.startswith(c) or c in name for c in CIBLES):
        print(f"  L{ln:>4}  {kind:9} {name}")

Invariant.lean : 71 declarations

  L 233  def       IsTricolorable
  L 276  instance  IsTricolorable.decidable
  L 310  theorem   triColorFoxCondition_iff_sum_mod_three
  L 687  theorem   tricolorable_forward_r1
  L 932  theorem   tricolorable_forward_r2_up
  L1060  theorem   r2_append_only_wall
  L1213  theorem   r3_determined_wall
  L1352  theorem   tricolorable_invariant_fails_under_pr1_model
  L3188  theorem   tricolorable_invariant_r2_connected
  L3474  theorem   tricolorable_invariant_r3_connected
  L3489  theorem   tricolorable_invariant


### Lecture du résultat
La chaîne expose la discipline de preuve du lake : `tricolorable_invariant` (le théorème-cible) est décomposé en bras `tricolorable_forward_r1` et `tricolorable_forward_r2_up`, chacun borné par des **murs nommés** (`r2_append_only_wall`, `r3_determined_wall`) — des `theorem` qui énoncent l'obligation restante au lieu de la cacher dans un `sorry` anonyme. La bi-implication R1 est complète (forward + backward, #11227).

## 4. Sorries réels vs prose — l'instrument juste

Le README du lake documente deux comptes : `grep -c sorry` naïf (compte la prose) et le mode `real` du CI (strippe commentaires, compte mot-bounded). Un `grep` naïf sur `Invariant.lean` rend 15 là où le réel est **2**. On mesure les deux.

In [4]:
# Les deux comptes, mesures sur les fichiers reels
def count_sorry(path, mode):
    text = path.read_text(encoding='utf-8')
    if mode == 'naive':
        return text.count('sorry')
    # mode real : strippe -- et /- -/, compte le mot bounded
    stripped = re.sub(r'/-([^-]|-[^/])*-/', ' ', text, flags=re.S)
    stripped = re.sub(r'--.*', ' ', stripped)
    return len(re.findall(r'\bsorry\b', stripped))

rows = []
for p in modules:
    rows.append([p.stem, count_sorry(p, 'naive'), count_sorry(p, 'real')])
rows.append(['TOTAL', sum(r[1] for r in rows), sum(r[2] for r in rows)])
import pandas as pd
df = pd.DataFrame(rows, columns=['module', 'grep naif', 'sorry reels'])
df

,module,grep naif,sorry reels
0,Basic,3,0
1,Conway,11,8
2,Invariant,8,1
3,Lidman,4,2
4,MathlibPrerequisites,2,0
5,Reidemeister,2,2
6,TOTAL,30,13


### Lecture du résultat
L'écart naive/réel illustre le piège documenté dans les règles du dépôt : les docstrings FR documentent précisément l'absence de preuve (« mur R2 », « restent 2 sorries ») — un `grep` naïf compte cette documentation. Le CI gate sur le compte réel (baseline 14). Un instrument qui surestime la dette d'un facteur 2,5 fausse l'arbitrage de ce qu'il faut prouver ensuite.

## 5. `Reidemeister.lean` — les moves et le corridor sémantique

Les moves R1/R2/R3 y sont des `Prop` explicites (`Reidemeister1`, `Reidemeister2`, ...) avec symétries prouvées. Le corridor #8696 (5 PRs mergées) y a remplacé le move-surgery `with` par des égalités de champs. Deux sorries réels subsistent : `reidemeister_theorem` ×2 (topologie PL des 3-variétés, hors Mathlib actuel — documenté INTRINSIC).

In [5]:
# Les moves Reidemeister et leurs proprietes
reid = decls_of(LAKE / 'Knots' / 'Reidemeister.lean')
print(f"Reidemeister.lean : {len(reid)} declarations\n")
for ln, kind, name in reid:
    tag = ''
    if name.startswith('Reidemeister'):
        tag = '  <-- move'
    print(f"  L{ln:>4}  {kind:9} {name}{tag}")

Reidemeister.lean : 36 declarations

  L  87  def       Reidemeister1  <-- move
  L  97  theorem   Reidemeister1.symm  <-- move
  L 145  def       Reidemeister1'  <-- move
  L 164  theorem   Reidemeister1'.implies_reidemeister1  <-- move
  L 217  def       PDCrossing.isRenameOf
  L 229  def       PDCrossing.hasEdge
  L 262  def       Reidemeister1Connected  <-- move
  L 286  theorem   reidemeister1Connected_satisfiable
  L 320  theorem   Reidemeister1Connected.numEdges_succ  <-- move
  L 328  theorem   Reidemeister1Connected.numCrossings_succ  <-- move
  L 337  theorem   Reidemeister1Connected.shares_edge  <-- move
  L 350  theorem   Reidemeister1Connected.crossings_eq  <-- move
  L 374  def       Reidemeister2  <-- move
  L 383  theorem   Reidemeister2.symm  <-- move
  L 414  def       PDCrossing.isDoubleRenameOf
  L 444  def       Reidemeister2Connected  <-- move
  L 466  theorem   reidemeister2Connected_satisfiable
  L 494  theorem   Reidemeister2Connected.numEdges_succ  <-- move
  

### Lecture du résultat
Chaque move a sa `.symm` prouvée (la symétrie n'est pas une hypothèse cachée). `Reidemeister1Connected` vient avec ses théorèmes de comptage (`numEdges_succ`, `numCrossings_succ`) : le modèle R1-connecté rend le move *localement vérifiable* — c'est ce qui a permis le corridor.

## 6. `MathlibPrerequisites.lean` — la feuille de route des 11 prérequis

Ce module ne prouve rien — et c'est son rôle. Chacune de ses 11 déclarations est un théorème **volontairement vide** (`True := trivial`) : ce qui compte n'est pas l'énoncé mais la docstring au-dessus, qui documente **ce que Mathlib ne sait pas encore faire** pour tel résultat de théorie des nœuds (Epic #2874, Phase 1). Trois tiers de difficulté annoncent trois horizons : Tier 1 accessible (les cibles Phase 2), Tier 2 modéré (Alexander, Jones), Tier 3 approfondi (le théorème de Reidemeister, Piccirillo, Lidman, Freedman — les mêmes résultats que Lean-17 raconte historiquement).

In [6]:
# Les 11 prerequis, extraits du fichier reel, avec tier / enjeu / reference
mp_path = LAKE / 'Knots' / 'MathlibPrerequisites.lean'
mp_text = mp_path.read_text(encoding='utf-8')

print(f"MathlibPrerequisites.lean : {len(decls_of(mp_path))} declarations, {count_sorry(mp_path, 'real')} sorry reel\n")

rows = []
tier, enjeu, ref = None, None, ''
for line in mp_text.splitlines():
    m = re.match(r'.*-!\s*## (Tier \d)\s*:\s*(\S+)', line)
    if m:
        tier = f"{m.group(1)} ({m.group(2)})"
        continue
    m = re.match(r'/--\s*(#\d+\s*:.*)', line)
    if m:
        enjeu = m.group(1)
        continue
    m = re.match(r'Reference\s*:\s*(.*)', line)
    if m:
        ref = m.group(1)
        continue
    dm = DECL_RE.match(line)
    if dm:
        rows.append([tier, dm.group(2), enjeu, ref])
        enjeu, ref = None, ''

for t, name, enjeu, ref in rows:
    print(f"  {t:22} {name:42} {enjeu}")
    if ref:
        print(f"  {'':22} {'':42} ref. {ref}")
print()
df_mp = pd.DataFrame(rows, columns=['tier', 'declaration', 'enjeu', 'reference'])
df_mp.groupby('tier', sort=False).size().rename('declarations').to_frame()

MathlibPrerequisites.lean : 11 declarations, 0 sorry reel

  Tier 1 (Accessible)    pd_wellformed_prerequisites                #1 : Bien-fondation des PD-codes
  Tier 1 (Accessible)    trefoil_tricolorable_prerequisites         #2 : Le trefoil est tricoloriable
  Tier 1 (Accessible)    unknot_not_tricolorable_prerequisites      #3 : Le noeud trivial n'est pas tricoloriable
  Tier 1 (Accessible)    tricolorable_invariant_prerequisites       #4 : La tricolorabilite est invariante sous R1, R2, R3
  Tier 2 (Modere)        reidemeister_formal_prerequisites          #5 : Mouvements de Reidemeister (description formelle)
                                                                    ref. shua/leanknot possede une formalisation partielle.
  Tier 2 (Modere)        alexander_polynomial_prerequisites         #6 : Polynome d'Alexander
                                                                    ref. Alexander (1928), Crowell & Fox (1963)
  Tier 2 (Modere)        jones_polynomial_prereq

,declarations
tier,
Tier 1 (Accessible),4
Tier 2 (Modere),3
Tier 3 (Approfondi),4


### Lecture du résultat

Onze déclarations, **0 sorry réel** : l'instrument du dépôt (`count_code_sorry.py`) les classe « vacuous markers » — ce ne sont pas de la dette de preuve, ce sont de la **documentation formalisée**.

Le **Tier 1** cible exactement la chaîne 3-colorabilité de §3 (bien-fondé des PD-codes, tricoloriable du trèfle, non-tricoloriable du nœud trivial, invariance R1/R2/R3) — la feuille de route a vieilli dans le bon sens : ce sont ces cibles que §3-§4 montrent en cours de réalisation (bras, murs nommés, sorry restants localisés).
Le **Tier 2** prolonge le corridor de §5 : description formelle des moves, polynôme d'Alexander via Burau/Fox, polynôme de Jones via le crochet de Kauffman.
Le **Tier 3** est l'horizon de recherche : le théorème de Reidemeister (isotopie ambiante ↔ moves) et le trio du nœud de Conway — Piccirillo (non lisse-slice), Freedman (topologiquement slice), Lidman (dénouement de 11n102) — les deux premiers inscrits au [Lean AI Leaderboard](https://lean-lang.org/eval/). Chronologie annoncée pour ce tier : « années à décennies ».

La feuille de route et l'histoire racontée par Lean-17 se répondent : ce que Lean-17 présente comme résultats mathématiques établis (Piccirillo 2020, Lidman 2026), ce module l'écrit comme ce que Mathlib devrait construire pour qu'ils deviennent des théorèmes Lean.

## 7. Le tableau de bord du lake

Synthèse : déclarations, sorries réels, et théorèmes-phare par module.

In [7]:
# Tableau de bord : declarations / theoremes / sorries reels par module
rows = []
for p in modules:
    ds = decls_of(p)
    thm = sum(1 for _, k, _ in ds if k in ('theorem', 'lemma'))
    rows.append([p.stem, len(ds), thm, count_sorry(p, 'real')])
df = pd.DataFrame(rows, columns=['module', 'declarations', 'theoremes', 'sorries reels'])
df['sorries/decl'] = (df['sorries reels'] / df['declarations']).round(2)
df

,module,declarations,theoremes,sorries reels,sorries/decl
0,Basic,34,14,0,0.00
1,Conway,15,5,8,0.53
2,Invariant,71,48,1,0.01
3,Lidman,4,2,2,0.50
4,MathlibPrerequisites,11,11,0,0.00
5,Reidemeister,36,22,2,0.06


### Lecture du résultat
`MathlibPrerequisites` porte le cadre sans sorry réel ; `Basic` est propre (0) ; la dette se concentre dans `Conway` (8) et le pair `Invariant`/`Reidemeister` (2+2). Le ratio sorries/declarations rend lisible où porte l'effort de preuve restant.

## 8. Le miroir i18n — byte-identity du code

La convention #4980 exige que seules les docstrings et commentaires diffèrent entre `Foo.lean` et `Foo_en.lean`. Vérification mesurée : on retire prose et commentaires des deux fichiers et on compare.

In [8]:
# Byte-identity via l'instrument canonique du depot
# (check_i18n_siblings.py : la byte-identity est MODULO les lignes qui
# differentent legitimement -- import/open/namespace _en, suffixes _en --
# un check naive du code brut rendrait des faux NON. On localise le repo
# en remontant depuis le cwd jusqu'au dossier scripts/.)
import subprocess, sys
from pathlib import Path
root = Path.cwd()
while not (root / "scripts" / "lean" / "check_i18n_siblings.py").exists():
    root = root.parent
r = subprocess.run(
    [sys.executable, str(root / "scripts" / "lean" / "check_i18n_siblings.py"), "knot_lean"],
    capture_output=True, text=True, cwd=str(root / "MyIA.AI.Notebooks" / "SymbolicAI" / "Lean"),
)
out = (r.stdout or r.stderr).strip().splitlines()
print(out[-1] if out else f"(aucune sortie, rc={r.returncode})")


7/7 pairs byte-identical | 0 consumer-pattern | 0 drift | 0 orphan | 0 unbuilt (0 whitelisted) | 0 half-done (advisory)


### Lecture du résultat
Le lake est en état i18n **drained** : 7/7 paires byte-identiques au sens canonique (modulo `import`/`open`/`namespace _en` et les suffixes `_en` d'identifiants — les seules lignes qui diffèrent légitimement), seules les docstrings diffèrent réellement. C'est la garantie que le sibling EN ne dérive jamais du FR. Un test naïf du code brut rendrait des faux négatifs — c'est exactement pourquoi le dépôt a un instrument canonique.

## Exercice 1 — Les murs nommés

Les murs (`r2_append_only_wall`, `r3_determined_wall`) sont la signature de la discipline de preuve du lake : énoncer l'obligation restante au lieu de la cacher. Retrouvez leur ligne exacte dans `Invariant.lean` et le texte de l'énoncé.

**Indice :** cherchez les `theorem` dont le nom contient `wall`. Attention, l'énoncé peut être sur plusieurs lignes — capturez aussi la ligne suivante.

**Étape 1 :** lister toutes les déclarations dont le nom contient `wall`.  
**Étape 2 :** pour chaque mur, afficher ses 2 premières lignes d'énoncé.

In [9]:
# Exercice a completer : localiser les murs nommes d'Invariant.lean
# Etape 1 : noms des declarations contenant 'wall'
# Etape 2 : pour chacune, les 2 premieres lignes suivant la declaration
pass

## Exercice 2 — Le compte qui fonde la baseline CI

Le CI gate sur **13 sorries réels** (lean-knot.yml, post-#11958). Recomptez avec l'instrument `real` ci-dessus et confrontez à la baseline.

**Indice :** la fonction `count_sorry(path, 'real')` est déjà écrite. Le résultat doit être comparé à la constante 13 — un écart dans un sens ou l'autre est un signal (régression ou baseline périmée).

**Étape 1 :** total real sur les six modules.  
**Étape 2 :** afficher la comparaison au format `total vs baseline : OK/ECART`.

In [10]:
# Exercice a completer : confrontez le compte real a la baseline CI (13)
BASELINE = 13
pass

## Exercice 3 — Votre premier mur

La discipline du lake : quand une preuve résiste, on ne met pas un `sorry` anonyme — on énonce le mur. Écrivez (en Python, pour l'analyse) un détecteur de sorries *anonymes* : un `sorry` dont le théorème enveloppant ne porte ni `wall` ni commentaire d'explication.

**Indice :** parcourez les lignes ; quand vous croisez `sorry` en mode réel, remontez au `theorem`/`def` englobant et vérifiez si son nom ou les 5 lignes au-dessus contiennent `wall` ou `--`.

**Étape 1 :** fonction `anonymous_sorries(path)` retournant la liste des noms de déclarations fautives.  
**Étape 2 :** l'appliquer aux six modules — le résultat attendu sur ce lake est une liste courte.

In [11]:
# Exercice a completer : detecteur de sorries anonymes
def anonymous_sorries(path):
    # retourne la liste des declarations portant un sorry sans mur nomme
    # ni commentaire d'explication a proximite
    return None  # TODO etudiant

pass

## Références

- Lake : [`knot_lean/`](knot_lean/) — README (état des sorries, corridor Reidemeister #8696, bi-implication R1 #11227)
- Lean-17a : [`Lean-17a-Knots-Conway-Proofs.ipynb`](Lean-17a-Knots-Conway-Proofs.ipynb) — l'histoire mathématique (Fox, Piccirillo, Lidman)
- Règles du dépôt : compte `sorry` réel via `scripts/lean/count_code_sorry.py` (jamais `grep -c`)
- Feuille de route : `Knots/MathlibPrerequisites.lean` (Epic #2874, Phase 1) — les 11 prérequis tier par tier
- Lean AI Leaderboard : [conway_knot_not_smoothly_slice](https://lean-lang.org/eval/problems/conway_knot_not_smoothly_slice/), [conway_knot_topologically_slice](https://lean-lang.org/eval/problems/conway_knot_topologically_slice/) — les cibles Tier 3
- Piccirillo (2020), *The Conway knot is not slice*, Annals — via Lean-17 §3
- Lidman (2026), unknotting number de 11n102 — via Lean-17 §5

## Conclusion

Le lake `knot_lean` n'est pas un décor : 171 déclarations réelles, une dette de preuve **ciblée** (13 sorries réels sur 4 modules, dont 10 documentés hors-portée Mathlib), une chaîne 3-colorabilité décomposée en bras et murs nommés, un miroir i18n byte-identique. Le compagnon mesure ce que Lean-17 raconte : la formalisation avance en discipline — chaque mur est un théorème, chaque sorry restant est nommé et localisé.